# Unconfoundedness: компактный production-процесс

Оценка эффекта бинарного воздействия по наблюдательным данным: **ATTE (ATT)** —
средний эффект среди получивших воздействие. Без EDA и графиков.
Для ATE по всей целевой популяции замените `score="ATTE"` на `score="ATE"`.

Предполагаем независимые наблюдения (одна строка на клиента), общий момент
начала наблюдения и одинаковый горизонт outcome. Все общие причины treatment
и outcome включены в X и измерены до воздействия; есть overlap, нет interference.
Диагностика и sensitivity не доказывают отсутствие скрытого confounding.

Ноутбук запускается в окружении с установленным `causalis`. Используем встроенные
CatBoost-модели и cross-fitting; свои learners и ручной расчёт статистик не нужны.

In [9]:
import json
import pandas as pd

from causalis.scenarios.unconfoundedness import IRM
from causalis.scenarios.unconfoundedness.dgp import obs_linear_26_dataset
from causalis.scenarios.unconfoundedness.refutation import (
    run_overlap_diagnostics,
    run_score_diagnostics,
    run_unconfoundedness_diagnostics,
    run_sensitivity_protocol,
)

## 1. Данные и заранее заданное правило решения

Генератор нужен только для воспроизводимого примера: возвращает `CausalData`
с outcome `y`, treatment `d` и пятью наблюдаемыми confounders, без oracle-столбцов.

Для production замените генератор своим контрактом:

```python
from causalis.data_contracts import CausalData

data = CausalData(
    df=df, user_id="user_id", treatment="d", outcome="y",
    confounders=["tenure_months", "avg_sessions_week", "spend_last_month",
                 "premium_user", "urban_resident"],
)
```

Состав X, benchmark-группу, направление и практически значимый порог выбирают
до оценки эффекта. Здесь порог — ноль в единицах outcome, проверяем положительный
эффект. Benchmark — история активности и трат: её исключение задаёт ориентир
силы возможного скрытого confounder.

`DESIGN_APPROVED=True` относится к дизайну синтетического примера. На реальных
данных значение должно отражать проверку adjustment set, временных окон,
отсутствия leakage и SUTVA; статистические флаги её не заменяют.

In [10]:
DESIGN_APPROVED = True
DECISION_THRESHOLD = 0.0
BENCHMARK_GROUPS = {"activity_and_spend": ["avg_sessions_week", "spend_last_month"]}

data = obs_linear_26_dataset(n=10_000, seed=42, include_oracle=False)

## 2. Оценка и доверительный интервал

Один `fit` и один `estimate`; 95% CI. Четыре folds и хранение диагностик включены
по умолчанию. `n_jobs=2` ограничивает параллелизм. Стандартный clipping propensity
на 0.01 защищает численные расчёты, но сам по себе не решает проблему overlap.

In [11]:
model = IRM(random_state=42, n_jobs=2).fit(data)
estimate = model.estimate(score="ATTE", alpha=0.05)
estimate.summary()

,value
field,
outcome,y
estimand,ATTE
model,IRM
value,"1.6168 (ci_abs: 1.2725, 1.9610)"
value_relative,"22.5033 (ci_rel: 17.1150, 27.8917)"
alpha,0.0500
p_value,0.0000
is_significant,True
n_treated,504


## 3. Проверки перед использованием результата

Три вызова проверяют поддержку и веса, ортогональность и влияние наблюдений,
баланс X после взвешивания. Сводки объединяем в одну таблицу.

В примере используется консервативное правило: все флаги в сводной таблице должны быть `GREEN`.
Неприменимые компоненты (например, g1 для ATTE) API не включает в эту таблицу.
`YELLOW`, `RED` и `NA` не проходят автоматический допуск и требуют разбора.
Это пример политики допуска, а не универсальные статистические критерии.

In [12]:
reports = {
    "overlap": run_overlap_diagnostics(data, estimate),
    "score": run_score_diagnostics(data, estimate),
    "balance": run_unconfoundedness_diagnostics(data, estimate),
}
diagnostic_summary = pd.concat(
    {name: report["summary"] for name, report in reports.items()}, names=["check"]
)
diagnostics_passed = diagnostic_summary["flag"].eq("GREEN").all().item()
diagnostic_summary

metric         value    flag
check                                                    
overlap 0           edge_0.01_below  5.430000e-02     RED
        1           edge_0.01_above  0.000000e+00     RED
        2           edge_0.02_below  1.783000e-01     RED
        3           edge_0.02_above  0.000000e+00     RED
        4                        KS  1.272649e-01   GREEN
        5                       AUC  5.730947e-01   GREEN
        6              support_rate  1.000000e+00   GREEN
        7         ESS_treated_ratio  3.801613e-01   GREEN
        8         ESS_control_ratio  9.922685e-01   GREEN
        9          tails_w1_q99/med  8.829223e+00  YELLOW
        10         tails_w0_q99/med  1.245461e+00   GREEN
        11      ATT_identity_relerr  2.205432e-03   GREEN
        12             clip_m_total  5.430000e-02     RED
        13                calib_ECE  1.220805e-02   GREEN
        14              calib_slope  3.050393e-01     RED
        15          calib_intercept -1.964239e+00     RED
score   0          psi_p99_over_med  5.297205e+01     RED
        1              psi_kurtosis  7.432988e+01     RED
        2                max_|t|_g0  8.258957e-01   GREEN
        3                 max_|t|_m  1.114283e+00   GREEN
        4             oos_max_abs_t  7.759613e-16   GREEN
balance 0           balance_max_smd  4.886099e-02   GREEN
        1   balance_frac_violations  0.000000e+00   GREEN

## 4. Чувствительность и итоговый статус

Один вызов протокола; внутри — дополнительный short-model refit на benchmark-группу.
`PASS` означает, что предпосылки допуска выполнены и bias-aware CI в основном
benchmark-сценарии целиком выше заданного порога. `FAIL` означает, что заявленный
вывод не прошёл правило; это не доказательство нулевого эффекта.

Протокол также рассчитывает stress и adversarial сценарии; они описывают запас
устойчивости, но не меняют основное правило PASS/FAIL. Сохраняем предупреждения:
в частности, устойчивость к повторным разбиениям cross-fitting здесь не проверяется.
Sensitivity запускается и при непройденной диагностике, но итог останется `FAIL`.

In [13]:
protocol = run_sensitivity_protocol(
    estimate, data,
    benchmark_groups=BENCHMARK_GROUPS,
    decision_threshold=DECISION_THRESHOLD,
    direction="positive",
    preconditions_passed=DESIGN_APPROVED and diagnostics_passed,
)
print(protocol["summary"])
print("Warnings:", protocol["warnings"])
protocol["primary"]

FAIL: the external design/overlap/nuisance preconditions did not pass; the causal claim is not established.
Warnings: ["Benchmark 'activity_and_spend' used DoubleML boundary calibration: cf_d_raw=-0.100475 was clipped to 0; rho used the DoubleML boundary fallback because both gain components were not strictly positive.", 'IRM currently uses n_rep=1; repeated cross-fitting stability must be assessed separately.']


,benchmark,benchmark_group,scenario,multiplier,r2_y,r2_d,rho,theta_long,theta_short,delta,relative_delta,theta_lower,theta_upper,ci_lower,ci_upper,rv,rva,scenario_valid,scenario_issue,passed
0,activity_and_spend,"(avg_sessions_week, spend_last_month)",primary,1.0,0.328521,0.0,1.0,1.616765,2.477844,-0.86108,0.532594,1.616765,1.616765,1.272504,1.961025,0.099477,0.078604,True,None,True


## 5. Результат для сервиса

`result_json` содержит численную оценку и CI, флаги проверок, параметры и итог
sensitivity. Диагностические массивы и обученная модель исключены.
`allow_nan=False` запрещает передачу нечисловых значений как валидного результата.
Для аудита подробные отчёты доступны в `reports` и `protocol`.

In [14]:
result_json = json.dumps({
    "estimate": estimate.model_dump(mode="json", exclude={"diagnostic_data"}),
    "design_approved": DESIGN_APPROVED,
    "diagnostics_passed": diagnostics_passed,
    "diagnostic_flags": {name: report["flags"] for name, report in reports.items()},
    "sensitivity": {
        "status": protocol["status"],
        "summary": protocol["summary"],
        "warnings": protocol["warnings"],
        "benchmark_groups": BENCHMARK_GROUPS,
        "decision_threshold": DECISION_THRESHOLD,
        "direction": protocol["direction"],
    },
}, ensure_ascii=False, allow_nan=False)

In [16]:
result_json

'{"estimate": {"estimand": "ATTE", "model": "IRM", "model_options": {"n_folds": 4, "n_rep": 1, "normalize_ipw": false, "overlap_policy": "clip", "overlap_threshold": 0.01, "overlap_n_dropped": 0, "random_state": 42, "n_jobs": 2, "std_error": 0.17564639134883914, "t_stat": 9.204656841178565, "se_approx_hajek": false, "se_approx_weight_norm": false}, "value": 1.6167647577574196, "ci_upper_absolute": 1.961025358815572, "ci_lower_absolute": 1.2725041566992672, "value_relative": 22.503329773916864, "ci_upper_relative": 27.89169659702256, "ci_lower_relative": 17.11496295081117, "alpha": 0.05, "p_value": 0.0, "is_significant": true, "n_treated": 504, "n_control": 9496, "treatment_mean": 8.801322660967715, "control_mean": 5.954581970194321, "adjusted_treatment_mean": null, "adjusted_control_mean": null, "outcome": "y", "treatment": "d", "confounders": ["tenure_months", "avg_sessions_week", "spend_last_month", "premium_user", "urban_resident"], "time": "2026-10-31"}, "design_approved": true, "d

In [ ]:
result_json